In [1]:
import sys
import os
import glob
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px

sys.path.append('..')
from src.database import init_db, get_connection
from src.processor import procesar_excel_brou

init_db()

# Buscar archivo
files = glob.glob("../data/raw/*.xls*")
if not files:
    print("❌ No se encontraron archivos.")
else:
    latest_file = max(files, key=os.path.getctime)
    df_nuevo = procesar_excel_brou(latest_file)
    # Creamos la columna categoría vacía para obligar a llenarla
    if 'categoria' not in df_nuevo.columns:
        df_nuevo['categoria'] = None
    print(f"✅ Archivo cargado. {len(df_nuevo)} movimientos detectados.")

--- DEBUG: Iniciando lectura de ../data/raw/Detalle_Movimiento_Cuenta (1).xls ---
--- DEBUG: ¡Tabla real encontrada en fila 31! ---
--- DEBUG: Columnas finales detectadas: ['Fecha', 'Descripción', 'Unnamed: 2', 'Número de documento', 'Asunto', 'Dependencia', 'Débito', 'Crédito']
--- DEBUG: Se procesaron 21 movimientos ---
✅ Archivo cargado. 21 movimientos detectados.


In [26]:
CATEGORIAS = ["Delivery","Super", "Fijos", "Ocio", "Transporte", "Salud", "Educación", "Sueldo", "Otros", " -- bug --"]

def procesar_categorias():
    clear_output()
    # Buscamos los que aún no tienen categoría
    pendientes = df_nuevo[df_nuevo['categoria'].isnull()]
    
    if not pendientes.empty:
        idx = pendientes.index[0]
        row = pendientes.iloc[0]
        
        print(f"--- CATEGORIZACIÓN OBLIGATORIA ({len(pendientes)} restantes) ---")
        print(f"Detalle: {row['descripcion']}")
        print(f"Monto: ${row['monto']}")
        
        selector = widgets.Dropdown(options=CATEGORIAS, description='Categoría:')
        boton = widgets.Button(description="Confirmar", button_style='success')
        
        def al_confirmar(b):
            df_nuevo.at[idx, 'categoria'] = selector.value
            procesar_categorias() # Salta al siguiente
            
        boton.on_click(al_confirmar)
        display(selector, boton)
    else:
        print("✅ ¡Todo categorizado! Ya puedes ejecutar la siguiente celda para guardar y graficar.")

procesar_categorias()

✅ ¡Todo categorizado! Ya puedes ejecutar la siguiente celda para guardar y graficar.


In [27]:
if df_nuevo['categoria'].isnull().any():
    print("❌ Error: Todavía hay movimientos sin categoría. Completa la celda anterior.")
else:
    conn = get_connection()
    try:
        # Limpiamos para evitar errores de duplicados y asegurar datos nuevos
        conn.execute("DELETE FROM movimientos")
        df_nuevo.to_sql("movimientos", conn, if_exists="append", index=False, method="multi")
        conn.commit()
        print("💾 Datos guardados en la base de datos.")
        
        # Gráfico
        df_total = pd.read_sql("SELECT * FROM movimientos", conn)
        df_total['monto'] = pd.to_numeric(df_total['monto'])
        df_gastos = df_total[df_total['monto'] < 0].copy()
        df_gastos['monto_abs'] = df_gastos['monto'].abs()
        
        fig = px.pie(
            df_gastos, 
            values='monto_abs', 
            names='categoria', 
            title='Resumen de Gastos por Categoría',
            hole=0.4,
            template="plotly_dark"
        )
        fig.show()
        
    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        conn.close()

💾 Datos guardados en la base de datos.


In [28]:
import pandas as pd
from src.database import get_connection

def consultar_db(tabla="movimientos", limite=20):
    conn = get_connection()
    try:
        # Consultamos la tabla deseada ordenada por fecha (si es movimientos)
        query = f"SELECT * FROM {tabla}"
        if tabla == "movimientos":
            query += " ORDER BY fecha DESC"
        query += f" LIMIT {limite}"
        
        df_consulta = pd.read_sql(query, conn)
        
        if df_consulta.empty:
            print(f"📭 La tabla '{tabla}' está vacía.")
        else:
            print(f"📊 Mostrando últimos {len(df_consulta)} registros de la tabla '{tabla}':")
            display(df_consulta)
            
            # Pequeño resumen si es la tabla de movimientos
            if tabla == "movimientos" and 'monto' in df_consulta.columns:
                print(f"\n💰 Total en esta vista: ${df_consulta['monto'].sum():.2f}")
                
    except Exception as e:
        print(f"❌ Error al consultar la tabla '{tabla}': {e}")
    finally:
        conn.close()

# --- EJECUCIÓN ---
# Cambia "movimientos" por "reglas" si querés ver tus categorías guardadas
consultar_db(tabla="movimientos", limite=50)

📊 Mostrando últimos 21 registros de la tabla 'movimientos':


,fecha,descripcion,monto,categoria,hash
0,Esta información es la que consta en los Siste...,,0.00,-- bug --,797433513240566215
1,2026-05-04 00:00:00,Comercio: DLO*PedidosYa Propin,-50.00,Delivery,18297504266440184759
2,2026-05-04 00:00:00,Comercio: DLO*PedidosYa Envio,-56.07,Delivery,1528549957141123243
3,2026-05-04 00:00:00,Comercio: DLO*PedidosYa Helade,-379.75,Delivery,5935310815070854216
4,2026-05-04 00:00:00,Comercio: DLO*PedidosYa Propin,-100.00,Delivery,17009870666861617922
5,2026-05-04 00:00:00,Comercio: DLO*PedidosYa Envio,-70.82,Super,7461135742591152002
6,2026-05-04 00:00:00,Comercio: DLO*PedidosYa Locos,-1287.46,Delivery,14289839871079085353
7,2026-05-04 00:00:00,Comercio: DLO*PedidosYa Propin,-30.00,Delivery,15148875007829001908
8,2026-05-04 00:00:00,Comercio: DLO*PedidosYa Envio,-48.20,Delivery,11067871429215834953
9,2026-05-04 00:00:00,Comercio: DLO*PedidosYa Sandwi,-287.13,Super,11045368651333818264



💰 Total en esta vista: $76537.76
